# Generar sample_df: Extracción de una muestra aleatoria de todos los CSV
Objetivo: Visitar todos los archivos CSV de la carpeta especificada, procesarlos en chunks,
y extraer una muestra aleatoria (reservoir sampling) de todos los datos. 
El resultado se guarda en un archivo "sample_df.csv" para usarlo en análisis posteriores.

In [2]:


import glob
import pandas as pd
import numpy as np
import os
import random

folder_name = "../outphys_beta_0403"

# ------------------ Variables Globales Modificables ------------------
FILE_PATTERN = f"{folder_name}/*.csv"   # Patrón de archivos CSV (ajústalo según tu estructura)
CHUNKSIZE = 10000                        # Número de filas a leer por chunk
MAX_SAMPLE_SIZE = 100000                 # Tamaño máximo de la muestra aleatoria
OUTPUT_FILENAME = "sample_df.csv"        # Nombre del archivo de salida para la muestra

# ------------------ Inicialización ------------------
files = glob.glob(FILE_PATTERN)
print("Número de archivos encontrados:", len(files))

reservoir = []  # Lista para almacenar la muestra
total_records = 0  # Contador global de registros procesados

# ------------------ Procesamiento de Archivos ------------------
for file in files:
    print(f"Procesando archivo: {os.path.basename(file)}")
    try:
        # Leer el archivo CSV en chunks
        for chunk in pd.read_csv(file, chunksize=CHUNKSIZE):
            # Reemplazar infinitos por NaN (opcional, según necesidad)
            chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
            # Si deseas eliminar filas con NaN, descomenta la siguiente línea:
            # chunk = chunk.dropna()
            if chunk.empty:
                continue
            # Convertir cada chunk a lista de registros (diccionarios)
            for record in chunk.to_dict(orient="records"):
                total_records += 1
                if total_records <= MAX_SAMPLE_SIZE:
                    reservoir.append(record)
                else:
                    # Con probabilidad MAX_SAMPLE_SIZE/total_records, reemplazar un elemento en el reservoir
                    s = random.randint(1, total_records)
                    if s <= MAX_SAMPLE_SIZE:
                        idx = random.randint(0, MAX_SAMPLE_SIZE - 1)
                        reservoir[idx] = record
    except Exception as e:
        print(f"Error al procesar {file}: {e}")

print(f"Total de registros procesados: {total_records}")
print(f"Tamaño final de la muestra: {len(reservoir)}")

# ------------------ Conversión y Guardado ------------------
sample_df = pd.DataFrame(reservoir)
# Guardar la muestra en un CSV
sample_df.to_csv(OUTPUT_FILENAME, index=False)
print(f"Muestra guardada en: {OUTPUT_FILENAME}")


Número de archivos encontrados: 1
Procesando archivo: beta_explore.csv
Total de registros procesados: 16032
Tamaño final de la muestra: 16032
Muestra guardada en: sample_df.csv
